In [ ]:
import numpy as np
from scipy import signal

# JEKOVA

### Jekova Filter

**2.2. Algorithm**

The analysis was applied on subsequent 10 s epochs. The general flow chart of the algorithm
is presented in figure 1. All analysis and test procedures were performed with the software
package MATLAB 6.0 (MathWorks, Inc.).

**2.2.1. Signal preprocessing filtration. The applied signal preprocessing included**

> (i) two successive first-order high-pass filters with 1 Hz cut-off frequency to suppress residual baseline drift;  
> (ii) a second-order 30 Hz Butterworth low-pass filter to reduce muscle noise, following the approach of Thakor et al (1990) and  
> (iii) a notch filter to eliminate powerline interference. 

The equivalent high-pass filter cut-off frequency of 1.4 Hz is slightly higher than the accepted bandwidth (0.67–30 Hz) for ‘monitor’ type ECG (IEC 62D/60601-2-27 1994). As for defibrillator monitors there is no strictly specified bandwidth (IEC Committee Draft 2001), practically a relatively high-frequency cut-off, up to 2 Hz is acceptable, since it does not attenuate VF or VT signals (Charbonnier 1994). In addition, this brought the advantage of faster recovery after high-amplitude noise and a defibrillation pulse artefact, as well as better suppression of residual baseline drift.

**2.2.2. Noise and asystoly detection.**

The first step of the algorithm is noise detection. It makes use of criteria for detection of abnormal signal amplitudes and slopes, uncharacteristic for ECG signals. The amplitude threshold is chosen according to the dynamic range of the
input amplifiers and analogue to digital (AD) converter to detect extreme artefacts (for example
AD converter saturation). The maximum slew rate limit above which a signal is considered ‘noise’ was set at 400 µV ms−1.
Signals with amplitudes below 150 µV are not analysed and classified as ‘Asystoly’.

**2.2.3. Band-pass digital filtration.**

The VF/VT detection method uses a band-pass
digital filter to pass the supraventricular complexes (normal sinus rhythm, atrial tachycardia,
atrial fibrillation, atrial flutter and sinus tachycardia) and ventricular complexes (premature
ventricular contractions and ventricular tachycardia) with frequencies up to 20 Hz and 14 Hz
respectively (Minami et al 1999). The filter has to suppress the ventricular fibrillation and
ventricular flutter peaks with frequency components below 7 Hz, according to Murray et al
(1985) and Clayton et al (1994), and less than 10 Hz, after Minami et al (1999). Therefore,
one can consider that the frequency range between 13 and 17 Hz contains the frequency
components of the non-shockable rhythm complexes and almost does not contain frequency
components of the shockable rhythms. Accordingly, we selected a central frequency at 15 Hz
with ±2 Hz bandwidth and designed a recursive filter with floating point precision coefficients.
Aiming at a simpler solution, convenient for embedding in an AED microprocessor system,
we preferred to use a digital filter with integer coefficients. A recursive filter with central
frequency at 14.6 Hz and bandwidth from 13 Hz to 16.5 Hz (−3 dB) was obtained by reducing
the floating point coefficients to integer coefficients. The filter equation (1), valid for 250 Hz
sampling frequency, was designed in consideration of its future real-time implementation:

$$FS[i] = (14*FS[i-1] - 7*FS[i-2] + (S[i] - S[i-2]) / 2) / 8$$

$$
FS_i = \frac{14\,FS_{i-1} - 7\,FS_{i-2} + \frac{S_i - S_{i-2}}{2}}{8}
$$

Equivalent form:

$$
FS_i = \frac{7}{4}FS_{i-1} - \frac{7}{8}FS_{i-2} + \frac{1}{16}\left(S_i - S_{i-2}\right)
$$

Here S[i] is a signal sample with index i; FS[i] is the filtered signal sample with index i.

In [ ]:
def JakovaFilter(y, W: int):
    hfc = 1.0
    lfc = 30.0

    ## High pass filter ###############
    
    ## First order IIR from Bilinear Transform ##
    # Prewarped analog cutoff (rad/s equivalent under BLT)
    # k = 2 * FS
    # wc = k * np.tan(np.pi * hfc / FS)
    # b0 = k / (k + wc)
    # b1 = -b0
    # a1 = (wc - k) / (k + wc)
    # b = [b0, b1]
    # a = [1, a1]

    ## First order Butterworth ##
    b, a = signal.butter(N=1, Wn=hfc, btype="highpass", fs=FS) # type: ignore
    y = signal.lfilter(b, a, y)   # first section
    y = signal.lfilter(b, a, y)   # second section
    ###################################

    ## Butter low pass 30 Hz ##########
    b, a = signal.butter(N=2, Wn=lfc, btype='low', fs=FS) # type: ignore
    y = signal.lfilter(b, a, y) # type: ignore
    ###################################

    ## Jekova Equation ################
    # y[i] = (14*y[i-1] - 7*y[i-2] + (x[i] - x[i-2])/2) / 8
    b = [1/16, 0, -1/16]
    a = [1, -28/16, 14/16]
    # y = signal.lfilter(b, a, y) * 8
    y = signal.lfilter(b, a, y) * 5
    ###################################

    return np.array(y)
pass #def